# Matched DiD Analysis (Virginia Data Centers)

This notebook runs weighted TWFE and event-study models using a matched panel dataset.

In [ ]:
import pandas as pd
from linearmodels.panel import PanelOLS

In [ ]:
# Load data (update path if needed)
FILE = 'va_did_panel.csv'
panel = pd.read_csv(FILE)

# Clean column names
panel.columns = panel.columns.str.strip().str.lower()

# Rename for consistency
panel = panel.rename(columns={
    'census_geoid': 'geoid',
    'first_entry_year': 'first_treat_year'
})

# Format variables
panel['geoid'] = panel['geoid'].astype(str).str.zfill(11)
panel['year'] = pd.to_numeric(panel['year'], errors='coerce')
panel['first_treat_year'] = pd.to_numeric(panel['first_treat_year'], errors='coerce')
panel['weight'] = pd.to_numeric(panel['weight'], errors='coerce')

for c in ['avg_total_lmp','avg_energy','avg_congestion','avg_loss','inefficient_pricing']:
    panel[c] = pd.to_numeric(panel[c], errors='coerce')

In [ ]:
# Treatment variables
panel['treated_it'] = panel['treated'].fillna(0).astype(int)
panel['event_time'] = pd.to_numeric(panel['event_time'], errors='coerce')

## Weighted TWFE

In [ ]:
panel_fe = panel.set_index(['geoid','year']).sort_index()

mod = PanelOLS.from_formula(
    'inefficient_pricing ~ 1 + treated_it + EntityEffects + TimeEffects',
    data=panel_fe,
    weights=panel_fe['weight']
)

res = mod.fit(cov_type='clustered', cluster_entity=True)
print(res.summary)

## Event Study

In [ ]:
event_map = {
    -4: 'event_m4',
    -3: 'event_m3',
    -2: 'event_m2',
     0: 'event_0',
     1: 'event_p1',
     2: 'event_p2',
     3: 'event_p3',
     4: 'event_p4'
}

for k,v in event_map.items():
    panel[v] = (panel['event_time'] == k).astype(int)

panel_fe = panel.set_index(['geoid','year']).sort_index()

event_vars = ' + '.join(event_map.values())

mod_es = PanelOLS.from_formula(
    f'inefficient_pricing ~ 1 + {event_vars} + EntityEffects + TimeEffects',
    data=panel_fe,
    weights=panel_fe['weight']
)

res_es = mod_es.fit(cov_type='clustered', cluster_entity=True)
print(res_es.summary)